# 2주차 3교시 · 같은 활동 기록을 넓은 표와 긴 표로 바꾸기

**학생용 복습완성본**

오늘의 질문: **같은 28개 기록을 비교하기 좋은 표와 계산하기 좋은 표로 바꿀 수 있을까요?**

오늘의 완성 결과: 4명×7일 넓은 표와 28행 긴 표를 만들고, 전체 활동표를 CSV와 Parquet로 저장합니다.

학생이 막히면 정답 전체를 바로 보여주지 말고, [자료구조] → [입력] → [한 줄씩 읽기] → [정상 결과] 순으로 힌트를 줍니다.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

def find_week2_data_dir():
    """노트북을 어느 교시 폴더에서 열어도 2주차 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "02_week2",
            base / "matchmaking_data" / "02_week2",
            base / "02_week2",
        ]
        for candidate in candidates:
            if (candidate / "activity_batches").exists():
                return candidate
    raise FileNotFoundError("02_week2 데이터 폴더를 찾지 못했습니다. python_lecture 프로젝트 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week2_data_dir()
WEEK2_DIR = DATA_DIR.parents[1] / "week2"
OUTPUT_DIR = WEEK2_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print("데이터 폴더:", DATA_DIR)
batch_files = sorted((DATA_DIR / "activity_batches").glob("activity_batch_*.csv"))
frames = []
for file_path in batch_files:
    one_batch = pd.read_csv(file_path)
    one_batch["source_file"] = file_path.name
    frames.append(one_batch)
activity = pd.concat(frames, ignore_index=True)


## Q1. 4명과 7일만 골라 표 모양을 눈으로 확인할 준비를 합니다

**먼저 예상하기**

- 회원 4명의 7일 기록은 왜 28행일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 30,000행 전체보다 눈으로 셀 수 있는 작은 표에서 모양 변화를 이해합니다.

**현재 자료구조** small은 28행×10열 DataFrame입니다.

**코드 읽기 도움** query 안의 @는 파이썬 변수 member_ids와 dates의 값을 조건문 안에서 사용한다는 표시입니다.

**정상 결과** (28, 10)

**오류가 나면** 행이 28이 아님. member_ids 4개와 dates 7개, query 조건을 각각 출력합니다.


In [ ]:
member_ids = ["M001", "M002", "M003", "M004"]
dates = sorted(activity["activity_date"].unique())[:7]
small = activity.query("member_id in @member_ids and activity_date in @dates")
print(small.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 회원 4명의 7일 기록은 왜 28행일까요?
- 기대 답: 한 행이 회원 한 명의 하루 기록이므로 4명×7일은 28행입니다.
- 그대로 말할 문장: “query 안의 @는 파이썬 변수 member_ids와 dates의 값을 조건문 안에서 사용한다는 표시입니다.”
- 결과 확인: (28, 10)
- 복구 순서: member_ids 4개와 dates 7개, query 조건을 각각 출력합니다.


## Q2. 회원은 행에 두고 날짜를 열로 펼쳐 비교합니다

**먼저 예상하기**

- 한 회원의 7일 앱 실행 수를 가로로 보려면 표를 어떻게 바꿔야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 날짜별 앱 실행 횟수를 회원별 가로 표로 확인합니다.

**현재 자료구조** wide는 index가 회원 4명, columns가 날짜 7개인 4×7 DataFrame입니다.

**코드 읽기 도움** pivot_table은 새 값을 만들기보다 기존 행을 행축과 열축에 다시 배치합니다. 같은 칸이 여러 개면 aggfunc로 합칩니다.

**정상 결과** (4, 7)

**오류가 나면** KeyError 또는 중복 조합 오류. index·columns·values에 실제 열 이름을 넣었는지 확인합니다.


In [ ]:
wide = small.pivot_table(
    index="member_id", columns="activity_date",
    values="app_opens", aggfunc="sum", fill_value=0
)
print(wide.shape)
display(wide)

### 강사 설명 메모

- 학생에게 물을 질문: 한 회원의 7일 앱 실행 수를 가로로 보려면 표를 어떻게 바꿔야 할까요?
- 기대 답: member_id를 행에, activity_date를 열에, app_opens를 칸의 값에 놓습니다.
- 그대로 말할 문장: “pivot_table은 새 값을 만들기보다 기존 행을 행축과 열축에 다시 배치합니다. 같은 칸이 여러 개면 aggfunc로 합칩니다.”
- 결과 확인: (4, 7)
- 복구 순서: index·columns·values에 실제 열 이름을 넣었는지 확인합니다.


## Q3. 날짜 열 7개를 다시 한 개의 날짜 열로 모읍니다

**먼저 예상하기**

- 넓은 표를 그래프나 집계에 쓰기 좋은 긴 표로 되돌리려면 어떻게 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 그래프·집계·파일 결합에 편한 긴 형태로 되돌립니다.

**현재 자료구조** long_again은 28행×3열 DataFrame입니다.

**코드 읽기 도움** melt는 여러 날짜 열의 이름을 activity_date 값으로, 각 셀의 숫자를 app_opens 값으로 내립니다.

**정상 결과** (28, 3)

**오류가 나면** member_id가 value 열로 녹아 들어감. reset_index 후 columns를 보고 id_vars 철자를 확인합니다.


In [ ]:
long_again = wide.reset_index().melt(
    id_vars="member_id",
    var_name="activity_date",
    value_name="app_opens"
)
print(long_again.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 넓은 표를 그래프나 집계에 쓰기 좋은 긴 표로 되돌리려면 어떻게 할까요?
- 기대 답: member_id는 그대로 두고 날짜 열 이름과 각 칸의 값을 두 개의 열로 모읍니다.
- 그대로 말할 문장: “melt는 여러 날짜 열의 이름을 activity_date 값으로, 각 셀의 숫자를 app_opens 값으로 내립니다.”
- 결과 확인: (28, 3)
- 복구 순서: reset_index 후 columns를 보고 id_vars 철자를 확인합니다.


## Q4. 체류 시간을 숫자로 정리하고 CSV와 Parquet로 저장합니다

**먼저 예상하기**

- 문자 '미측정'이 섞인 열을 그대로 Parquet로 저장해도 될까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** Parquet는 한 열의 자료형을 분명하게 저장하므로 '미측정'과 숫자가 섞인 열을 먼저 정리하고, CSV와 Parquet의 차이를 경험합니다.

**현재 자료구조** activity_export는 원본을 복사해 stay_time_min만 숫자로 정리한 30,000행×10열 DataFrame입니다.

**코드 읽기 도움** Parquet는 자료형을 저장하므로 한 열에 '미측정' 문자와 숫자가 섞이면 먼저 숫자 변환이 필요합니다.

**정상 결과** True True

**오류가 나면** ArrowInvalid 또는 ImportError: pyarrow. stay_time_min을 to_numeric(errors='coerce')로 정리했는지와 pyarrow 설치 여부를 확인합니다.


In [ ]:
activity_export = activity.copy()
activity_export["stay_time_min"] = pd.to_numeric(activity_export["stay_time_min"], errors="coerce")
csv_path = OUTPUT_DIR / "activity_merged.csv"
parquet_path = OUTPUT_DIR / "activity_merged.parquet"
activity_export.to_csv(csv_path, index=False)
activity_export.to_parquet(parquet_path, index=False)
print(csv_path.exists(), parquet_path.exists())

### 강사 설명 메모

- 학생에게 물을 질문: 문자 '미측정'이 섞인 열을 그대로 Parquet로 저장해도 될까요?
- 기대 답: 먼저 stay_time_min을 숫자로 바꾸고, 바꿀 수 없는 '미측정'은 NaN으로 표시해야 합니다.
- 그대로 말할 문장: “Parquet는 자료형을 저장하므로 한 열에 '미측정' 문자와 숫자가 섞이면 먼저 숫자 변환이 필요합니다.”
- 결과 확인: True True
- 복구 순서: stay_time_min을 to_numeric(errors='coerce')로 정리했는지와 pyarrow 설치 여부를 확인합니다.


## Q5. 저장한 파일을 다시 읽어 크기와 자료형을 확인합니다

**먼저 예상하기**

- 저장 성공 메시지만 보고 파일을 믿어도 될까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 저장이 성공했다는 메시지보다 다시 읽은 결과가 같은지 검증합니다.

**현재 자료구조** csv_back과 parquet_back은 각각 30,000행×10열 DataFrame입니다.

**코드 읽기 도움** read_parquet은 Parquet 파일을 DataFrame으로 다시 읽습니다.

**정상 결과** (30000, 10) (30000, 10)

**오류가 나면** shape 불일치. 저장하기 전 activity.shape와 두 경로를 출력합니다.


In [ ]:
csv_back = pd.read_csv(csv_path)
parquet_back = pd.read_parquet(parquet_path)
assert csv_back.shape == activity_export.shape
assert parquet_back.shape == activity_export.shape
print(csv_back.shape, parquet_back.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 저장 성공 메시지만 보고 파일을 믿어도 될까요?
- 기대 답: 아니요. 다시 읽어서 행과 열 수, 일부 값과 자료형을 확인해야 합니다.
- 그대로 말할 문장: “read_parquet은 Parquet 파일을 DataFrame으로 다시 읽습니다.”
- 결과 확인: (30000, 10) (30000, 10)
- 복구 순서: 저장하기 전 activity.shape와 두 경로를 출력합니다.


## 마무리 확인

- [ ] 오늘 결과가 **4명×7일 넓은 표와 28행 긴 표를 만들고, 전체 활동표를 CSV와 Parquet로 저장합니다.**와 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
